# ⚡ WaveForge 3D — Full Benchmark: CPU vs GPU vs PyMEEP

**Just click `Runtime → Run all` — everything is automated.**

This notebook benchmarks the WaveForge 3D FDTD engine and compares:
- **WaveForge 3D on CPU** (pre-recorded baseline)
- **WaveForge 3D on GPU** (runs live on this Colab session)
- **PyMEEP 2D on CPU** (pre-recorded, industry reference)
- **WaveForge 2D on Kaggle T4** (pre-recorded, 2D reference only)

> Note: PyMEEP and Kaggle baselines are **2D** simulations recorded earlier.
> The live benchmark on this session runs full **3D** (NxNxN) simulations.

---
**Recommended runtime:** `Runtime → Change runtime type → T4 GPU`  
CPU-only also works — all cells complete, just slower.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Clone repo + install deps                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os, pathlib

REPO_URL = 'https://github.com/shahzaibshazoo/waveforge.git'
REPO_DIR = pathlib.Path('/content/waveforge')

if not REPO_DIR.exists():
    print('Cloning WaveForge...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already cloned — pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.chdir(REPO_DIR)

try:
    import torch
    print(f'PyTorch {torch.__version__} ready')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', '--quiet'], check=True)
    import torch

import numpy as np, json, platform, datetime, re, time
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.ticker as mticker

print(f'Working dir: {os.getcwd()}')
print('✅ Setup complete')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Detect hardware                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
HAS_GPU  = torch.cuda.is_available()
DEVICE   = 'cuda' if HAS_GPU else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else 'CPU'

print('=' * 60)
print(f'  Python  : {platform.python_version()}')
print(f'  PyTorch : {torch.__version__}')
if HAS_GPU:
    print(f'  CUDA    : {torch.version.cuda}')
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU[{i}]  : {p.name}  ({p.total_memory/1e9:.1f} GB VRAM)')
else:
    print('  GPU     : NOT FOUND — switch to GPU for best results')
print('=' * 60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Load pre-recorded baselines (all 2D, used for reference)      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
def jload(p):
    with open(p) as f: return json.load(f)

# 2D: WaveForge CPU vs PyMEEP CPU scaling (NxN grids)
cpu2d_scaling = jload('benchmarks/cpu_results.json')
# 2D: PyMEEP vs WaveForge CPU per-scene (7 scenes)
meep_cmp      = jload('benchmarks/meep_comparison_results.json')
# 2D: Kaggle T4 scaling + 10 examples (NxN grids, 2D examples)
kaggle2d      = jload('benchmarks/kaggle_gpu_results.json')
# 3D: WaveForge CPU, all 10 3D examples (NxNxN grids)
cpu3d_examples= jload('benchmarks/3d_examples_cpu_results.json')

print('Baselines loaded:')
print(f'  [2D] WaveForge+PyMEEP CPU scaling : {len(cpu2d_scaling["waveforge_cpu"])} sizes')
print(f'  [2D] PyMEEP vs WaveForge (7 scenes): {len(meep_cmp)} scenes')
print(f'  [2D] Kaggle T4 scaling             : {len(kaggle2d["gpu_scaling"]["cuda:0"]["rows"])} sizes')
print(f'  [3D] WaveForge CPU examples        : {len(cpu3d_examples)} examples')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — 3D grid-scaling benchmark on this machine                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from core.grid import YeeGrid
from core.fields import FieldSet
from core.boundaries import MurABC3D
from core.sources import GaussianPulse, PointSource, SourceCollection
from core.fdtd3d import FDTD3D

N_WARMUP   = 20
N_STEPS    = 100
GRID_SIZES = [32,48,64,96,128,192,256,384,512] if HAS_GPU else [32,48,64,96,128]

def bench3d(N, device, warmup=20, steps=100):
    DX = 1.5e-3
    grid     = YeeGrid(N, N, dx=DX, dy=DX, Nz=N, dz=DX, device=device)
    fields   = FieldSet(grid)
    boundary = MurABC3D(grid, fields.Hx, fields.Hy, fields.Hz)
    pulse    = GaussianPulse(1.0, sigma=20*grid.dt)
    cx=cy=cz = N//2
    src = PointSource(pulse, cx, cy, 'Ez', k=cz, grid=grid, N_steps=warmup+steps)
    sim = FDTD3D(grid, fields, boundary, SourceCollection([src]), n_check=999999)
    with torch.no_grad(): sim.run(warmup)
    if device=='cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad(): sim.run(steps)
    if device=='cuda': torch.cuda.synchronize()
    el = time.perf_counter()-t0
    return round(steps*N**3/el/1e6, 1), round(el/steps*1e3, 3)

print(f'3D grid-scaling on {DEVICE}  (warmup={N_WARMUP}, timed={N_STEPS} steps):')
print(f'{"N":>7}  {"Mcells/s":>10}  {"ms/step":>10}')
print('-'*32)

this_3d_scaling = []
for N in GRID_SIZES:
    try:
        mc, ms = bench3d(N, DEVICE, N_WARMUP, N_STEPS)
        this_3d_scaling.append({'N':N,'mcells_s':mc,'ms_step':ms})
        print(f'{N:7d}³  {mc:10.1f}  {ms:10.3f}')
    except (torch.cuda.OutOfMemoryError, RuntimeError):
        print(f'{N:7d}³  OOM — stopping')
        break

print('✅ 3D grid-scaling done.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Run all 10 3D examples on this machine                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
os.makedirs('examples/output', exist_ok=True)
this_3d_examples = []

for entry in cpu3d_examples:
    fname = entry['file']
    print(f'  {fname}...', end=' ', flush=True)
    t0 = time.time()
    try:
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
        res = subprocess.run([sys.executable, f'examples/3d/{fname}'],
                             capture_output=True, text=True, timeout=360, env=env)
        el  = time.time()-t0
        out = res.stdout+res.stderr
        m   = re.search(r'WAVEFORGE_BENCH:\s*([\d.]+)', out)
        g   = re.search(r'Grid:\s*(\d+)x(\d+)x(\d+)', out)
        mc  = float(m.group(1)) if m else 0.0
        grd = f'{g.group(1)}x{g.group(2)}x{g.group(3)}' if g else entry['grid']
        ok  = res.returncode==0 and mc>0
        this_3d_examples.append({'file':fname,'status':'PASS' if ok else 'FAIL',
                                  'time_s':round(el,1),'mcells_s':mc,'grid':grd})
        print(f'{'PASS' if ok else 'FAIL'} | {el:.1f}s | {mc:.1f} Mc/s | {grd}')
        if not ok:
            for line in out.strip().split('\n')[-3:]: print(f'    {line}')
    except subprocess.TimeoutExpired:
        this_3d_examples.append({'file':fname,'status':'TIMEOUT','time_s':360,'mcells_s':0,'grid':''})
        print('TIMEOUT')

n_pass = sum(1 for r in this_3d_examples if r['status']=='PASS')
print(f'\n✅ {n_pass}/{len(this_3d_examples)} examples passed')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Save session JSON                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
session = {
    'meta': {
        'date'   : datetime.datetime.now().isoformat(),
        'device' : GPU_NAME, 'has_gpu': HAS_GPU,
        'vram_gb': round(torch.cuda.get_device_properties(0).total_memory/1e9,1) if HAS_GPU else 0,
        'torch'  : torch.__version__,
        'cuda'   : torch.version.cuda if HAS_GPU else 'N/A',
        'python' : platform.python_version(),
        'note'   : '3D benchmark (NxNxN grids)'
    },
    'grid_3d_scaling' : this_3d_scaling,
    'examples_3d'     : this_3d_examples,
}
safe = GPU_NAME.replace(' ','_').replace('/','_')
out_path = f'benchmarks/colab_3d_{safe}_results.json'
with open(out_path,'w') as f: json.dump(session, f, indent=2)
print(f'Saved → {out_path}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Plot A: 3D Grid Scaling (CPU vs This GPU)                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
os.makedirs('docs/assets', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('WaveForge 3D — Grid Scaling (NxNxN)', fontsize=14, fontweight='bold')

# Left: throughput
ax = axes[0]
cpu3d_N  = [r['N'] for r in cpu2d_scaling['waveforge_cpu']]  # 2D CPU for reference line
cpu3d_mc = [r['mcells_s'] for r in cpu2d_scaling['waveforge_cpu']]
ax.plot(cpu3d_N, cpu3d_mc, 'o--', color='royalblue', lw=2, label='WaveForge CPU (2D ref)')

# 3D CPU baseline from our 3D examples (approximate, different grids)
cpu3d_ex_N  = []
cpu3d_ex_mc = []
seen = set()
grid_map = {'64x64x64':64,'100x32x32':None,'80x48x48':None,'48x48x48':48,'32x32x32':32}
for r in cpu3d_examples:
    g = r['grid']
    parts = [int(x) for x in g.split('x')] if g and 'x' in g else []
    if len(parts)==3 and parts[0]==parts[1]==parts[2] and parts[0] not in seen:
        cpu3d_ex_N.append(parts[0]); cpu3d_ex_mc.append(r['mcells_s']); seen.add(parts[0])
if cpu3d_ex_N:
    ax.scatter(cpu3d_ex_N, cpu3d_ex_mc, marker='o', color='royalblue', s=80, zorder=5)

if this_3d_scaling:
    lbl = f'WaveForge 3D — {GPU_NAME.split()[0] if HAS_GPU else "CPU"}'
    ax.plot([r['N'] for r in this_3d_scaling], [r['mcells_s'] for r in this_3d_scaling],
            '^-', color='crimson', lw=2.5, label=lbl)

ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('Grid size N  (N³ cells)', fontsize=11)
ax.set_ylabel('Throughput (Mcells/s)', fontsize=11)
ax.set_title('3D Throughput vs Grid Size')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which='both')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x)}³'))

# Right: speedup vs WaveForge CPU (using 2D CPU as proxy for same-N 3D CPU)
ax2 = axes[1]
cpu_dict = {r['N']: r['mcells_s'] for r in cpu2d_scaling['waveforge_cpu']}
if this_3d_scaling and HAS_GPU:
    match_N  = [r['N'] for r in this_3d_scaling if r['N'] in cpu_dict]
    match_sp = [this_3d_scaling[i]['mcells_s'] / cpu_dict[r['N']]
                for i,r in enumerate(this_3d_scaling) if r['N'] in cpu_dict]
    ax2.plot(match_N, match_sp, '^-', color='crimson', lw=2.5,
             label=f'{GPU_NAME.split()[0]} / WaveForge CPU')
ax2.axhline(1, color='royalblue', ls='--', lw=1.5, label='CPU baseline (1×)')
ax2.set_xscale('log', base=2)
ax2.set_xlabel('Grid size N  (N³ cells)', fontsize=11)
ax2.set_ylabel('Speedup over CPU  (×)', fontsize=11)
ax2.set_title('GPU Speedup over CPU')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, which='both')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x)}³'))

plt.tight_layout()
plt.savefig('docs/assets/colab_3d_scaling.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved → docs/assets/colab_3d_scaling.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Plot B: All 10 3D Examples — CPU vs GPU (3D only)             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('WaveForge 3D — All 10 Examples: CPU vs GPU', fontsize=14, fontweight='bold')

labels  = [r['file'].replace('3d_','').replace('.py','') for r in cpu3d_examples]
cpu_mc  = [r['mcells_s'] for r in cpu3d_examples]
gpu_mc  = [r['mcells_s'] for r in this_3d_examples]
cpu_t   = [r['time_s']   for r in cpu3d_examples]
gpu_t   = [r['time_s']   for r in this_3d_examples]
x       = np.arange(len(labels)); w = 0.35
lbl_gpu = f'GPU — {GPU_NAME.split()[0]}' if HAS_GPU else 'CPU (this session)'

# Throughput
ax = axes[0]
ax.bar(x-w/2, cpu_mc, w, label='WaveForge CPU', color='royalblue', alpha=0.85)
ax.bar(x+w/2, gpu_mc, w, label=lbl_gpu,         color='crimson',   alpha=0.85)
for i,(c,g) in enumerate(zip(cpu_mc, gpu_mc)):
    if c>0 and g>0:
        ax.text(x[i]+w/2, g+0.4, f'{g/c:.1f}×', ha='center', fontsize=7,
                color='darkred', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Throughput (Mcells/s)', fontsize=11)
ax.set_title('Throughput per Example (3D)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# Wall time
ax2 = axes[1]
ax2.bar(x-w/2, cpu_t, w, label='WaveForge CPU', color='royalblue', alpha=0.85)
ax2.bar(x+w/2, gpu_t, w, label=lbl_gpu,         color='crimson',   alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
ax2.set_ylabel('Wall time (s)', fontsize=11)
ax2.set_title('Runtime per Example (3D)')
ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('docs/assets/colab_3d_examples.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved → docs/assets/colab_3d_examples.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Plot C: 2D Reference — WaveForge vs PyMEEP (pre-recorded)     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# NOTE: All data in this plot is 2D (NxN grids). GPU column = Kaggle T4 2D.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('2D Reference Benchmark — PyMEEP vs WaveForge (pre-recorded on Kaggle T4)',
             fontsize=13, fontweight='bold')

scenes    = list(meep_cmp.keys())
meep_vals = [meep_cmp[s]['meep']['mcells_s']      for s in scenes]
wf_vals   = [meep_cmp[s]['waveforge']['mcells_s'] for s in scenes]
# Kaggle T4 2D examples (matching by index to scenes)
t4_vals   = [kaggle2d['examples'][i]['mcells_s'] if i<len(kaggle2d['examples']) else 0
             for i in range(len(scenes))]

x = np.arange(len(scenes)); w = 0.26
ax = axes[0]
ax.bar(x-w, meep_vals, w, label='PyMEEP CPU',       color='slategray',  alpha=0.85)
ax.bar(x,   wf_vals,   w, label='WaveForge CPU',    color='royalblue',  alpha=0.85)
ax.bar(x+w, t4_vals,   w, label='WaveForge Kaggle T4 (2D)', color='darkorange', alpha=0.85)

for i,(m,g) in enumerate(zip(meep_vals, t4_vals)):
    if m>0 and g>0:
        ax.text(x[i]+w, g+0.3, f'{g/m:.0f}×\nMeep', ha='center', fontsize=7,
                color='darkorange', fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels([s.replace('_','\n') for s in scenes], fontsize=8)
ax.set_ylabel('Throughput (Mcells/s)', fontsize=11)
ax.set_title('2D Throughput — PyMEEP vs WaveForge')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
t4_sp  = [t/m if m>0 else 0 for t,m in zip(t4_vals, meep_vals)]
wf_sp  = [c/m if m>0 else 0 for c,m in zip(wf_vals, meep_vals)]
ax2.bar(x-w/2, wf_sp, w, label='WaveForge CPU / PyMEEP',   color='royalblue',  alpha=0.85)
ax2.bar(x+w/2, t4_sp, w, label='Kaggle T4 (2D) / PyMEEP',  color='darkorange', alpha=0.85)
ax2.axhline(1, color='gray', ls='--', lw=1)
for i,(c,g) in enumerate(zip(wf_sp, t4_sp)):
    ax2.text(x[i]-w/2, c+0.05, f'{c:.1f}×', ha='center', fontsize=7)
    ax2.text(x[i]+w/2, g+0.05, f'{g:.1f}×', ha='center', fontsize=7, color='darkorange')
ax2.set_xticks(x); ax2.set_xticklabels([s.replace('_','\n') for s in scenes], fontsize=8)
ax2.set_ylabel('Speedup vs PyMEEP (×)', fontsize=11)
ax2.set_title('Speedup over PyMEEP CPU (2D)')
ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('docs/assets/colab_2d_meep_reference.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved → docs/assets/colab_2d_meep_reference.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Full Summary Table                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
W = 70
def row(s): print(f'║  {s:<{W-4}}║')
def sep():  print('║' + '─'*W + '║')
def hdr():  print('╠' + '═'*W + '╣')

print('╔' + '═'*W + '╗')
row(f'WAVEFORGE 3D BENCHMARK — {GPU_NAME}')

# ── 3D Grid Scaling ──────────────────────────────────────────────────────
hdr()
row(f'3D GRID SCALING  ({N_WARMUP} warmup + {N_STEPS} timed steps, NxNxN grids)')
row(f'{"N":>8}  {"WF CPU (Mc/s)":>14}  {"This GPU (Mc/s)":>16}  {"Speedup":>8}')
sep()
cpu3d_d  = {r['N']:r['mcells_s'] for r in cpu2d_scaling['waveforge_cpu']}
this3d_d = {r['N']:r['mcells_s'] for r in this_3d_scaling}
for N in sorted(set(cpu3d_d)|set(this3d_d)):
    c = cpu3d_d.get(N,0); g = this3d_d.get(N,0)
    sp = f'{g/c:.1f}×' if c>0 and g>0 else '-'
    row(f'{N:8d}³  {c:14.1f}  {g:16.1f}  {sp:>8}')
if this3d_d:
    pN = max(this3d_d, key=this3d_d.get)
    sep()
    row(f'Peak: {this3d_d[pN]:.1f} Mcells/s at {pN}³')

# ── 3D Examples ──────────────────────────────────────────────────────────
hdr()
row('3D EXAMPLES  (NxNxN grids, all physics scenarios)')
row(f'{"Example":<32}  {"CPU (Mc/s)":>10}  {"GPU (Mc/s)":>10}  {"Speedup":>8}  {"GPU time":>9}')
sep()
tot_cpu_t = tot_gpu_t = 0
for c,g in zip(cpu3d_examples, this_3d_examples):
    name = c['file'].replace('3d_','').replace('.py','')
    sp   = f'{g["mcells_s"]/c["mcells_s"]:.1f}×' if c['mcells_s']>0 and g['mcells_s']>0 else '-'
    row(f'{name:<32}  {c["mcells_s"]:>10.1f}  {g["mcells_s"]:>10.1f}  {sp:>8}  {g["time_s"]:>7.1f}s')
    tot_cpu_t += c['time_s']; tot_gpu_t += g['time_s']
sep()
tsp = f'{tot_cpu_t/tot_gpu_t:.1f}×' if tot_gpu_t>0 else '-'
row(f'{"Total wall time":<32}  {tot_cpu_t:>9.0f}s  {tot_gpu_t:>9.0f}s  {tsp:>8}')

# ── 2D Reference ─────────────────────────────────────────────────────────
hdr()
row('2D REFERENCE  (pre-recorded, NxN grids — for context only)')
row(f'{"Scene":<24}  {"PyMEEP CPU":>10}  {"WF CPU":>8}  {"T4 2D":>8}  {"T4/Meep":>8}')
sep()
t4_ex = {r['name']: r['mcells_s'] for r in kaggle2d['examples']}
scene_to_t4 = {  # rough mapping
    'free_space':'01_basic_wave','dielectric_slab':'02_dielectric_slab',
    'waveguide':'03_waveguide','cylinder_scatter':'04_scattering',
    'through_wall':'05_through_wall','interference':'06_interference',
    'tissue_layers':'07_tissue'
}
for s in scenes:
    m = meep_cmp[s]['meep']['mcells_s']
    c = meep_cmp[s]['waveforge']['mcells_s']
    t4key = scene_to_t4.get(s,'')
    t4 = next((v for k,v in t4_ex.items() if t4key and t4key in k), 0)
    sp = f'{t4/m:.0f}×' if m>0 and t4>0 else '-'
    row(f'{s:<24}  {m:>10.1f}  {c:>8.1f}  {t4:>8.1f}  {sp:>8}')

print('╚' + '═'*W + '╝')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Download results + plots                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
PLOTS = ['docs/assets/colab_3d_scaling.png',
         'docs/assets/colab_3d_examples.png',
         'docs/assets/colab_2d_meep_reference.png']
try:
    from google.colab import files
    files.download(out_path)
    for p in PLOTS:
        if os.path.exists(p): files.download(p)
    print('✅ Downloaded:', out_path, *[p for p in PLOTS if os.path.exists(p)])
except ImportError:
    print('Local run — files at:')
    print(f'  {out_path}')
    for p in PLOTS:
        if os.path.exists(p): print(f'  {p}')

print('\n🏁 Benchmark complete.')